In [1]:
import numpy as np
from cnn.naive_cnn import Layer, Conv, BatchNorm, ReLU, Pooling, Flatten, Dense, Softmax, NeuralNet, Optimizer, Results
import time

In [2]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [4]:
data_dir = "../cifar-10-batches-py"

X_train = []
y_train = []

for i in range(1, 6):
    batch = unpickle(f"{data_dir}/data_batch_{i}")
    X_train.append(batch[b"data"])
    y_train.append(batch[b"labels"])

X_train = np.concatenate(X_train, axis=0)
y_train = np.concatenate(y_train)

test_batch = unpickle(f"{data_dir}/test_batch")

X_test = test_batch[b"data"]
y_test = np.array(test_batch[b"labels"])

X_train = X_train.reshape(-1, 3, 32, 32)
X_test = X_test.reshape(-1, 3, 32, 32)

X_train_norm = X_train / 255
X_test_norm = X_test / 255

X_train_norm = X_train_norm.astype(np.float32)
X_test_norm = X_test_norm.astype(np.float32)


print(X_train_norm.shape)
print(X_test_norm.shape)
print(y_train.shape)
print(y_test.shape)

(50000, 3, 32, 32)
(10000, 3, 32, 32)
(50000,)
(10000,)


In [5]:
naive_NN = NeuralNet()
naive_NN.layers = [Conv(32, 3, 3, 3, 1, 1), BatchNorm(32, 4), ReLU(), Conv(32, 32, 3, 3, 1, 1), BatchNorm(32, 4), ReLU(), Pooling(2, 2, 2), Conv(64, 32, 3, 3, 1, 1), BatchNorm(64, 4), ReLU(), Conv(64, 64, 3, 3, 1, 1), 
             BatchNorm(64, 4), ReLU(), Pooling(2, 2, 2), Conv(128, 64, 3, 3, 1, 1), BatchNorm(128, 4), ReLU(), Pooling(2, 2, 2), Flatten(), Dense(2048, 256), BatchNorm(256, 2), ReLU(), Dense(256, 10), Softmax()]

alpha = 0.001

In [7]:
X_batch = X_train_norm[:1]
y_batch = y_train[:1]

#Forward
start = time.perf_counter()

A_batch = naive_NN.forward(X_batch)

forward_time = time.perf_counter() - start


#Backward
start = time.perf_counter()

naive_NN.backward(A_batch, y_batch)

backward_time = time.perf_counter() - start


#Update
start = time.perf_counter()

naive_NN.update(alpha, 1)

update_time = time.perf_counter() - start


print(f"Forward: {forward_time} seconds")
print(f"Backward: {backward_time} seconds")
print(f"Update: {update_time} seconds")

total = forward_time + backward_time + update_time

print(f"Total: {total} seconds")

Forward: 57.90668509993702 seconds
Backward: 113.03928509983234 seconds
Update: 0.014867299934849143 seconds
Total: 170.9608374997042 seconds
